# Question 2: WordPiece Tokenization Algorithm (2 Merges)

**Course:** Natural Language Processing (AI357)  
**Institution:** Sardar Vallabhbhai National Institute of Technology (SVNIT), Surat  

---

### Problem Statement:
Consider the following corpus:
```
low lower low lower lowest
low lowest low lowest
low new
new new newer newer
new
```
Apply the **WordPiece algorithm for 2 merges**. What will be the tokenization of the word **`newest`**?  
**Marks:** $[2 + 2 + 2 = 6]$


## 1. WordPiece Algorithm Formulation

Unlike Byte Pair Encoding (BPE), which merges the pair with the highest raw frequency, **WordPiece** chooses the pair $(u, v)$ that maximizes the language model score:

$$\text{score}(u, v) = \frac{\text{count}(u, v)}{\text{count}(u) \times \text{count}(v)}$$

In WordPiece representation:
- The first character of a word is written normally (e.g., `'l'`).
- Any non-initial character is prefixed with `'##'` (e.g., `'##o'`, `'##w'`).


In [1]:
import pandas as pd
from collections import Counter

corpus_lines = [
    "low lower low lower lowest",
    "low lowest low lowest",
    "low new",
    "new new newer newer",
    "new"
]

# Step 1: Count word frequencies in the corpus
word_frequencies = Counter()
for line in corpus_lines:
    word_frequencies.update(line.split())

df_words = pd.DataFrame(list(word_frequencies.items()), columns=["Word", "Corpus Frequency"]).sort_values("Corpus Frequency", ascending=False)
print(f"Total word tokens in corpus: {sum(word_frequencies.values())}")
df_words


Total word tokens in corpus: 16


,Word,Corpus Frequency
0,low,5
3,new,4
2,lowest,3
1,lower,2
4,newer,2


In [2]:
# Step 2: Initialize words as character pieces with '##' for continuations
def split_word_to_pieces(word):
    return [word[0]] + ["##" + char for char in word[1:]]

word_pieces = {word: split_word_to_pieces(word) for word in word_frequencies}

print("Initial piece representation for each vocabulary word:")
for word, pieces in sorted(word_pieces.items()):
    print(f"  {word:7s} (freq = {word_frequencies[word]}): {pieces}")


Initial piece representation for each vocabulary word:
  low     (freq = 5): ['l', '##o', '##w']
  lower   (freq = 2): ['l', '##o', '##w', '##e', '##r']
  lowest  (freq = 3): ['l', '##o', '##w', '##e', '##s', '##t']
  new     (freq = 4): ['n', '##e', '##w']
  newer   (freq = 2): ['n', '##e', '##w', '##e', '##r']


In [3]:
# Functions to count pieces, pairs, and compute WordPiece scores
def get_piece_and_pair_frequencies(current_pieces, word_counts):
    piece_counts = Counter()
    pair_counts = Counter()
    for word, pieces in current_pieces.items():
        freq = word_counts[word]
        for piece in pieces:
            piece_counts[piece] += freq
        for i in range(len(pieces) - 1):
            pair_counts[(pieces[i], pieces[i + 1])] += freq
    return piece_counts, pair_counts

def calculate_wordpiece_table(piece_counts, pair_counts):
    rows = []
    scores = {}
    for pair, pair_cnt in pair_counts.items():
        left_p, right_p = pair
        c_left = piece_counts[left_p]
        c_right = piece_counts[right_p]
        score = pair_cnt / (c_left * c_right)
        scores[pair] = score
        rows.append({
            "Pair (u, v)": f"('{left_p}', '{right_p}')",
            "count(u, v)": pair_cnt,
            "count(u)": c_left,
            "count(v)": c_right,
            "WordPiece Score": score
        })
    df_table = pd.DataFrame(rows).sort_values("WordPiece Score", ascending=False)
    return scores, df_table

def apply_merge_to_vocabulary(current_pieces, pair_to_merge, merged_subword):
    new_word_pieces = {}
    for word, pieces in current_pieces.items():
        updated = []
        i = 0
        while i < len(pieces):
            if i + 1 < len(pieces) and (pieces[i], pieces[i + 1]) == pair_to_merge:
                updated.append(merged_subword)
                i += 2
            else:
                updated.append(pieces[i])
                i += 1
        new_word_pieces[word] = updated
    return new_word_pieces

print("Scoring functions ready.")


Scoring functions ready.


In [4]:
# Step 3: Compute Merge 1
piece_counts_1, pair_counts_1 = get_piece_and_pair_frequencies(word_pieces, word_frequencies)
scores_1, df_scores_1 = calculate_wordpiece_table(piece_counts_1, pair_counts_1)

print("=== Candidate Pairs for Merge 1 ===")
display(df_scores_1)

best_score_1 = max(scores_1.values())
best_pair_1 = sorted([pair for pair, s in scores_1.items() if s == best_score_1])[0]
new_piece_1 = best_pair_1[0] + best_pair_1[1].replace("##", "", 1)

print(f"\n--> WINNER FOR MERGE 1: {best_pair_1} with Score = {best_score_1:.6f}")
print(f"--> Merged Subword Token: '{new_piece_1}'\n")

word_pieces = apply_merge_to_vocabulary(word_pieces, best_pair_1, new_piece_1)

print("Vocabulary representation after Merge 1:")
for word, pieces in sorted(word_pieces.items()):
    print(f"  {word:7s}: {pieces}")


=== Candidate Pairs for Merge 1 ===


,"Pair (u, v)","count(u, v)",count(u),count(v),WordPiece Score
5,"('##s', '##t')",3,3,3,0.333333
0,"('l', '##o')",10,10,10,0.100000
4,"('##e', '##s')",3,13,3,0.076923
3,"('##e', '##r')",4,13,4,0.076923
6,"('n', '##e')",6,6,13,0.076923
1,"('##o', '##w')",10,10,16,0.062500
2,"('##w', '##e')",7,16,13,0.033654
7,"('##e', '##w')",6,13,16,0.028846



--> WINNER FOR MERGE 1: ('##s', '##t') with Score = 0.333333
--> Merged Subword Token: '##st'

Vocabulary representation after Merge 1:
  low    : ['l', '##o', '##w']
  lower  : ['l', '##o', '##w', '##e', '##r']
  lowest : ['l', '##o', '##w', '##e', '##st']
  new    : ['n', '##e', '##w']
  newer  : ['n', '##e', '##w', '##e', '##r']


In [5]:
# Step 4: Compute Merge 2
piece_counts_2, pair_counts_2 = get_piece_and_pair_frequencies(word_pieces, word_frequencies)
scores_2, df_scores_2 = calculate_wordpiece_table(piece_counts_2, pair_counts_2)

print("=== Candidate Pairs for Merge 2 ===")
display(df_scores_2)

best_score_2 = max(scores_2.values())
best_pair_2 = sorted([pair for pair, s in scores_2.items() if s == best_score_2])[0]
new_piece_2 = best_pair_2[0] + best_pair_2[1].replace("##", "", 1)

print(f"\n--> WINNER FOR MERGE 2: {best_pair_2} with Score = {best_score_2:.6f}")
print(f"--> Merged Subword Token: '{new_piece_2}'\n")

word_pieces = apply_merge_to_vocabulary(word_pieces, best_pair_2, new_piece_2)

print("Vocabulary representation after Merge 2:")
for word, pieces in sorted(word_pieces.items()):
    print(f"  {word:7s}: {pieces}")


=== Candidate Pairs for Merge 2 ===


,"Pair (u, v)","count(u, v)",count(u),count(v),WordPiece Score
0,"('l', '##o')",10,10,10,0.100000
4,"('##e', '##st')",3,13,3,0.076923
3,"('##e', '##r')",4,13,4,0.076923
5,"('n', '##e')",6,6,13,0.076923
1,"('##o', '##w')",10,10,16,0.062500
2,"('##w', '##e')",7,16,13,0.033654
6,"('##e', '##w')",6,13,16,0.028846



--> WINNER FOR MERGE 2: ('l', '##o') with Score = 0.100000
--> Merged Subword Token: 'lo'

Vocabulary representation after Merge 2:
  low    : ['lo', '##w']
  lower  : ['lo', '##w', '##e', '##r']
  lowest : ['lo', '##w', '##e', '##st']
  new    : ['n', '##e', '##w']
  newer  : ['n', '##e', '##w', '##e', '##r']


In [6]:
# Step 5: Tokenize the word 'newest'
target_word = "newest"
initial_tokens = split_word_to_pieces(target_word)

print(f"Target Word: '{target_word}'")
print(f"1. Initial split into characters: {initial_tokens}")

# Apply Merge 1: ('##s', '##t') -> '##st'
tokens_m1 = []
i = 0
while i < len(initial_tokens):
    if i + 1 < len(initial_tokens) and (initial_tokens[i], initial_tokens[i+1]) == best_pair_1:
        tokens_m1.append(new_piece_1)
        i += 2
    else:
        tokens_m1.append(initial_tokens[i])
        i += 1
print(f"2. After applying Merge 1 {best_pair_1} -> '{new_piece_1}': {tokens_m1}")

# Apply Merge 2: ('l', '##o') -> 'lo'
tokens_m2 = []
i = 0
while i < len(tokens_m1):
    if i + 1 < len(tokens_m1) and (tokens_m1[i], tokens_m1[i+1]) == best_pair_2:
        tokens_m2.append(new_piece_2)
        i += 2
    else:
        tokens_m2.append(tokens_m1[i])
        i += 1
print(f"3. After applying Merge 2 {best_pair_2} -> '{new_piece_2}': {tokens_m2}")

print(f"\n--> FINAL TOKENIZATION OF '{target_word}': {tokens_m2}")


Target Word: 'newest'
1. Initial split into characters: ['n', '##e', '##w', '##e', '##s', '##t']
2. After applying Merge 1 ('##s', '##t') -> '##st': ['n', '##e', '##w', '##e', '##st']
3. After applying Merge 2 ('l', '##o') -> 'lo': ['n', '##e', '##w', '##e', '##st']

--> FINAL TOKENIZATION OF 'newest': ['n', '##e', '##w', '##e', '##st']


## Summary of Answers

1. **Merge 1 [2 Marks]:**
   - Pair merged: `('##s', '##t')` $\to$ `'##st'`
   - WordPiece Score: $\frac{3}{3 \times 3} = \mathbf{0.333333}$

2. **Merge 2 [2 Marks]:**
   - Pair merged: `('l', '##o')` $\to$ `'lo'`
   - WordPiece Score: $\frac{10}{10 \times 10} = \mathbf{0.100000}$

3. **Tokenization of 'newest' [2 Marks]:**
   - Initial characters: `['n', '##e', '##w', '##e', '##s', '##t']`
   - After 2 merges: **`['n', '##e', '##w', '##e', '##st']`**
